# 06 - Distributed Geospatial Processing with Apache Sedona

Demonstrates the *processing framework* leg of the Big Data solution design (alongside the data lake/warehouse storage design and the ingestion notebooks in `scripts/download/`). Uses [Apache Sedona](https://sedona.apache.org/) - a geometry-aware extension of Spark - rather than plain PySpark, since the workload here is inherently spatial: a distributed spatial join between the analysis grid and administrative boundaries, plus a Spark SQL aggregation over the feature table.

Spark runs in local mode, without a cluster. It reads the fact table from HDFS and writes the Parquet outputs back to it, using the HDFS in `infra/hdfs` when it is running and the local `datasets/` folder otherwise. The input comes from `03_build_analysis_grid.ipynb` and `04_zonal_statistics.ipynb`. The raster zonal-statistics pipeline itself is untouched - Sedona is used here for the vector/tabular side only.

In [1]:
# add scripts/download to sys.path so we can `from common import ...`
import os, sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath("."))
if NOTEBOOK_DIR.name != "preprocessing":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "scripts" / "preprocessing"
os.chdir(NOTEBOOK_DIR)

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
DOWNLOAD_DIR = PROJECT_ROOT / "scripts" / "download"
if str(DOWNLOAD_DIR) not in sys.path:
    sys.path.insert(0, str(DOWNLOAD_DIR))

print("cwd:", os.getcwd())

cwd: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\scripts\preprocessing


## 1. Configuration & Sedona session

First run needs internet access once: `spark.jars.packages` pulls the Sedona/geotools JARs from Maven Central and caches them locally (can take a minute or two the first time, instant after that). If session creation fails with a "Java not found" style error, check that `java -version` works in a plain terminal (a JDK, not just a JRE, must be on PATH or `JAVA_HOME` set).

Spark on Windows also needs `winutils.exe`/`hadoop.dll` even in pure local mode (a quirk of Hadoop's `Shell` class on Windows, unrelated to the HDFS in `infra/hdfs`) - otherwise session creation fails with `HADOOP_HOME and hadoop.home.dir are unset`. The next cell points `HADOOP_HOME` at `tools/hadoop-3.3.6/` (binaries from the well-known [cdarlint/winutils](https://github.com/cdarlint/winutils) repo, matching Hadoop 3.3.6 which Spark 3.5.x bundles) - no system-wide install, no admin rights.

The session also sets the HDFS client options for `infra/hdfs`. `lake_path` (`scripts/download/common.py`) picks HDFS while its NameNode answers on `localhost:9000` and the local `datasets/` folder otherwise. `LAKE_URI=local` forces the local disk.


In [2]:
import os
import sys

from sedona.spark import SedonaContext

# Windows-only: point Spark's bundled Hadoop Shell class at the local winutils.exe/hadoop.dll
# (see markdown above) instead of a system-wide HADOOP_HOME - keeps this self-contained to the repo.
HADOOP_HOME = PROJECT_ROOT / "tools" / "hadoop-3.3.6"
if os.name == "nt" and (HADOOP_HOME / "bin" / "winutils.exe").exists():
    os.environ["HADOOP_HOME"] = str(HADOOP_HOME)
    os.environ["PATH"] = str(HADOOP_HOME / "bin") + os.pathsep + os.environ["PATH"]

# Pin the Python workers Spark spawns to this exact interpreter - with several conda
# envs on this machine, letting Spark resolve "python" from PATH can launch the wrong
# one (missing pyspark/pandas), which shows up as "Python worker failed to connect back".
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Pinned versions verified compatible as of 2026-09 (see
# https://sedona.apache.org/latest/setup/install-python/ if these ever need bumping -
# the three version numbers below must stay in sync with each other).
SEDONA_VERSION = "1.9.1"
SPARK_SCALA = "3.5_2.12"
GEOTOOLS_VERSION = "1.9.1-33.5"

config = (
    SedonaContext.builder()
    .appName("geoai-suitability-sedona-demo")
    .master("local[6]")   # at most 6 cores, the same CPU budget as the modeling notebooks (never all of them)
    # force the driver to bind to loopback only - otherwise Spark tries to bind a
    # network-visible socket and Windows Firewall pops up an "Allow access?" dialog
    # that silently hangs any non-interactive (or unattended) run
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    # HDFS client settings: the DataNode of infra/hdfs is published on localhost; harmless for local files
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
    .config("spark.hadoop.dfs.replication", "2")
    .config(
        "spark.jars.packages",
        f"org.apache.sedona:sedona-spark-{SPARK_SCALA}:{SEDONA_VERSION},"
        f"org.datasyslab:geotools-wrapper:{GEOTOOLS_VERSION}",
    )
    .config(
        "spark.jars.repositories",
        "https://artifacts.unidata.ucar.edu/repository/unidata-all",
    )
    .getOrCreate()
)
sedona = SedonaContext.create(config)
sedona.sparkContext.setLogLevel("WARN")
print("Spark version:", sedona.version)

D:\Softwares\Anaconda\Miniconda\envs\geoai-suitability\Lib\site-packages\sedona\spark\utils\geometry_serde.py:133: UserWarning: Cannot load geomserde_speedup, fallback to general python implementation. Reason: geos_c DLL not found in D:\Softwares\Anaconda\Miniconda\envs\geoai-suitability\Lib\site-packages\[S|s]hapely.libs
  warn(


Spark version: 3.5.9


## 2. Load the grid and province boundaries into Sedona

Reuses the already-verified `geopandas` readers (same as `03_build_analysis_grid.ipynb`) rather than Sedona's own file readers - lower risk, and geometries just need to cross the bridge into Spark as WKT.

In [3]:
import geopandas as gpd
from pyspark.sql.functions import expr

from common import RAW_DIR, PROCESSED_DIR

GRID_PATH = PROCESSED_DIR / "grid" / "grid_1km.gpkg"
PROVINCES_PATH = RAW_DIR / "boundaries" / "indonesia_adm1.geojson"

grid_gdf = gpd.read_file(GRID_PATH)  # EPSG:32748
provinces_gdf = gpd.read_file(PROVINCES_PATH).to_crs(grid_gdf.crs)  # match the grid's CRS

grid_pdf = grid_gdf[["grid_id", "geometry"]].copy()
grid_pdf["wkt"] = grid_pdf.geometry.apply(lambda g: g.wkt)

provinces_pdf = provinces_gdf[["shapeName", "geometry"]].copy()
provinces_pdf["wkt"] = provinces_pdf.geometry.apply(lambda g: g.wkt)

grid_sdf = sedona.createDataFrame(grid_pdf[["grid_id", "wkt"]]).withColumn(
    "geometry", expr("ST_GeomFromWKT(wkt)")
)
provinces_sdf = sedona.createDataFrame(provinces_pdf[["shapeName", "wkt"]]).withColumn(
    "geometry", expr("ST_GeomFromWKT(wkt)")
)

grid_sdf.createOrReplaceTempView("grid")
provinces_sdf.createOrReplaceTempView("provinces")

print(f"grid: {grid_sdf.count()} cells, provinces: {provinces_sdf.count()} polygons")

grid: 9333 cells, provinces: 34 polygons


## 3. Distributed spatial join - which province does each grid cell fall in?

A polygon-polygon spatial join (`ST_Intersects`) across all 9,333 cells x 34 provinces - exactly the kind of operation Sedona's spatial indexing is built to accelerate. Since the grid was built from the Banten AOI, this also doubles as a sanity check: almost every cell should resolve to Banten, with maybe a sliver at the border resolving to a neighboring province.

In [4]:
spatial_join = sedona.sql("""
    SELECT p.shapeName AS province, COUNT(*) AS n_cells
    FROM grid g
    JOIN provinces p
    ON ST_Intersects(g.geometry, p.geometry)
    GROUP BY p.shapeName
    ORDER BY n_cells DESC
""")
spatial_join.show(10, truncate=False)

+------------------------------+-------+
|province                      |n_cells|
+------------------------------+-------+
|Banten                        |9333   |
|West Java                     |91     |
|Jakarta Special Capital Region|27     |
+------------------------------+-------+



## 4. Spark SQL aggregation - best crop per grid cell

A simplified version of the argmax rule from `05_eda.ipynb` (pure argmax over the ten `gaez1k_index_*` columns, i.e. GAEZ v5 at 1 km from notebook 12, no "not recommended"/"tied" threshold logic) - it exists here to demonstrate a genuine Spark SQL aggregation over the feature table, not to reproduce the final crop recommendation label (see the dashboard, `scripts/dashboard/app.py`, for that). Updated from the original 4 crops to the full 10 (notebook 08 added oil palm, rubber, sugarcane, coffee, cacao, coconut) - with only 4, maize and soybean never won anywhere, so this undercounted the real picture - and from the ~9 km to the 1 km GAEZ product (notebook 12). A crop without a 1 km value in a cell (land GAEZ marks as no cropland) cannot win there; cells with no value for any crop are labelled `Water` (GAEZ class 10).

In [5]:
from pyspark.sql import functions as F

from common import lake_base, lake_path

FEATURES_PATH = lake_path("processed", "features", "tabular_features.csv")   # HDFS when infra/hdfs is running, else local
print("fact table read from:", FEATURES_PATH)
features_sdf = sedona.read.csv(FEATURES_PATH, header=True, inferSchema=True)

CROPS = ["rice", "maize", "cassava", "soybean", "oilpalm", "rubber", "sugarcane", "coffee", "cacao", "coconut"]
crop_names = {f"gaez1k_index_{c}": c.capitalize() for c in CROPS}   # GAEZ v5 at 1 km (notebook 12)
gaez_cols = list(crop_names.keys())

best_value = F.greatest(*[F.col(c) for c in gaez_cols])   # skips nulls; null only if every crop is null
best_crop_expr = None
for c in gaez_cols:
    cond = F.col(c) == best_value
    best_crop_expr = (
        F.when(cond, F.lit(crop_names[c])) if best_crop_expr is None
        else best_crop_expr.when(cond, F.lit(crop_names[c]))
    )
best_crop_expr = best_crop_expr.otherwise(F.lit("Water"))   # no 1 km value for any crop: GAEZ class 10

features_sdf = features_sdf.withColumn("best_crop", best_crop_expr)

features_sdf.groupBy("best_crop").count().orderBy(F.desc("count")).show()

+---------+-----+
|best_crop|count|
+---------+-----+
|  Oilpalm| 4293|
|  Coconut| 2707|
|   Rubber| 1440|
|   Coffee|  374|
|     Rice|  329|
|    Cacao|  138|
|  Cassava|   26|
|    Water|   23|
|    Maize|    3|
+---------+-----+



## 5. Save outputs (partitioned Parquet)

Partitioning by `best_crop` on write is the standard Big Data pattern for letting downstream queries prune to only the partitions (crops) they need instead of scanning everything.

The files go where the fact table came from: `/lake/processed/spark_output` on HDFS when it is running, `datasets/processed/spark_output` otherwise. The dashboard reads them from HDFS when it can.


In [6]:
from common import ensure_dir

SPARK_OUT = lake_path("processed", "spark_output")
if not lake_base():
    ensure_dir(PROCESSED_DIR / "spark_output")   # only the local disk needs the folder created

spatial_join.coalesce(1).write.mode("overwrite").parquet(SPARK_OUT + "/grid_by_province")
features_sdf.write.mode("overwrite").partitionBy("best_crop").parquet(SPARK_OUT + "/features_by_best_crop")

print(f"Saved outputs under {SPARK_OUT}")

Saved outputs under D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\datasets\processed\spark_output


## 6. Stop the Spark session

In [7]:
sedona.stop()